# ARCH 6131 · Scale：Jackson Heights + Corona plot筛选

个人仓库：Peterrr294/URBAN-DATA。

本笔记本是作业工作框架，不是已完成的分析。研究单位是有边界、可编号的plot；Northern Blvd/BQE仅为参考。先关联至少5套有实际分析作用的数据，再设定可解释的逐层规则，最后生成Index、Items以及Rhino/Grasshopper文件。不得为了凑条目数量倒推阈值。

教师方法：[S05](https://github.com/ivorywang-1005/intro-to-urban-data/blob/main/Scale/S05_Putting_It_All_Together.ipynb) → [S07](https://github.com/ivorywang-1005/intro-to-urban-data/blob/main/Scale/S07_Analyzing_proximity_with_apply_functions.ipynb) → [S13](https://github.com/ivorywang-1005/intro-to-urban-data/blob/main/Scale/S13_Joins_and_Aggregation.ipynb) → [S06](https://github.com/ivorywang-1005/intro-to-urban-data/blob/main/Scale/S06_Organizing_and_presenting_data.ipynb)。不复制答案；教师示例中的邮编/面积排序不用于最终catalog。


## 1. Colab设置与数据目录

GitHub保存代码和说明，Google Drive保存大体积输入和结果。先在Colab登录并挂载自己的Drive。不需要在代码里填写GitHub token。修改后可在Colab菜单“文件 → 在GitHub中保存副本”选择自己的仓库。


In [ ]:
%pip -q install geopandas pyarrow rhino3dm matplotlib
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
import json, math
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import rhino3dm

ROOT = Path("/content/drive/MyDrive/ARCH6131_Scale")
RAW, OUT = ROOT / "data/raw", ROOT / "outputs"
RAW.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)
CRS_LOCAL = 2263  # US survey feet，与教师教程一致
BUFFER_FEET = 660  # 示例分析半径，不是步行网络距离；需写明选择依据
print("数据放入:", RAW)


## 2. 准备输入（真实文件，不含示例假数据）

必需：`boundary.geojson`（明确研究边界）、`plots.geojson`（MapPLUTO或自绘候选地块，唯一`plot_id`）。候选plots应保留原始完整形状；仅选择与研究边界相交的plots，不将裁切碎片当成新地块。

可选文件：`crashes.geojson`（预先按时间筛选）、`food_stores.geojson`、`buildings.geojson`、`age_tracts.geojson`、`health_tracts.geojson`、`traffic_sites.geojson`。年龄/健康文件必须已关联真实统计区边界。交通数据先按测站/方向/日期汇总；不要把每一条15分钟记录当一个测站。下载来源和处理限制见仓库`DATA_SOURCES.md`。

桥下人工单元须标明边界来源；同一plot的MultiPolygon不是多个独立Items。每个输入缺失值均不得自动解释为0。


In [ ]:
def load_layer(name):
    path = RAW / name
    if not path.exists():
        raise FileNotFoundError(f"缺少输入文件：{path}")
    layer = gpd.read_file(path)
    if layer.crs is None:
        raise ValueError(f"{name}缺少CRS，先核实再设置")
    if layer.empty or layer.geometry.isna().any() or layer.geometry.is_empty.any():
        raise ValueError(f"{name}为空或含空几何")
    if not layer.geometry.is_valid.all():
        raise ValueError(f"{name}含无效几何，请检查修复并记录方法")
    return layer.to_crs(CRS_LOCAL)

boundary = load_layer("boundary.geojson")
plots = load_layer("plots.geojson")
if "plot_id" not in plots:
    raise ValueError("先在plots中建立唯一plot_id：税务地块可用BBL，自绘单元用独立前缀")
plots["plot_id"] = plots["plot_id"].astype(str).str.strip()
assert plots.plot_id.ne("").all() and plots.plot_id.is_unique, "plot_id必须非空且唯一"
assert plots.geom_type.isin(["Polygon", "MultiPolygon"]).all(), "plot必须是面"
area_of_interest = boundary.geometry.union_all()
plots = plots.loc[plots.intersects(area_of_interest)].copy().reset_index(drop=True)
assert len(plots), "研究范围内没有候选plots"
plots["plot_area_sqft"] = plots.area
print("初始候选plots:", len(plots))
display(plots.drop(columns="geometry").head())
plots.plot(figsize=(10, 10), edgecolor="black", linewidth=.2)


## 3. S07 + S13：每个plot关联周边对象数量

缓冲区统计是几何邻近，不是实际步行可达性。只有已核查完整的对象名录，零匹配才可标0。缺文件记NA；事故坐标缺失和商店名录遗漏需在报告中记录。一个事件可能同时落入多个plot缓冲区，因此不能把各plot数量相加作为区域唯一事件总数。


In [ ]:
metrics = plots.copy()
buffers = plots[["plot_id", "geometry"]].copy()
buffers.geometry = buffers.buffer(BUFFER_FEET)
coverage = []

for filename, field in [
    ("crashes.geojson", "crash_count_buffer"),
    ("food_stores.geojson", "food_store_count_buffer"),
]:
    if not (RAW / filename).exists():
        metrics[field] = pd.NA
        coverage.append({"file": filename, "status": "missing", "note": "不可填0"})
        continue
    layer = load_layer(filename)
    matches = gpd.sjoin(buffers, layer[["geometry"]], how="left", predicate="intersects")
    counts = matches.groupby("plot_id")["index_right"].count()
    metrics[field] = metrics.plot_id.map(counts).fillna(0).astype(int)
    coverage.append({"file": filename, "status": "loaded", "records": len(layer)})

display(pd.DataFrame(coverage))
display(metrics.drop(columns="geometry").head())


## 4. 统计区属性关联（年龄／健康）

先在GeoJSON中统一列名：年龄`age65_pct`、健康`adult_asthma_pct`，均使用0–100百分数；边界编号`GEOID`。本例使用plot内部代表点匹配单个tract，只说明其所属统计区背景，不估计服务人口，不是地块居民实测。跨tract的大plot需要另外审查。


In [ ]:
representatives = plots[["plot_id", "geometry"]].copy()
representatives.geometry = representatives.representative_point()

for filename, field in [
    ("age_tracts.geojson", "age65_pct"),
    ("health_tracts.geojson", "adult_asthma_pct"),
]:
    if not (RAW / filename).exists():
        metrics[field] = pd.NA
        continue
    tracts = load_layer(filename)
    for required in ["GEOID", field]:
        if required not in tracts:
            raise ValueError(f"{filename}缺少{required}")
    tracts["GEOID"] = tracts.GEOID.astype(str)
    assert tracts.GEOID.is_unique, "每个tract必须一行，先筛选年份/指标/估计类型"
    tracts[field] = pd.to_numeric(tracts[field], errors="coerce")
    assert tracts[field].dropna().between(0, 100).all(), "比例应为0–100"
    matched = gpd.sjoin(representatives, tracts[["GEOID", field, "geometry"]],
                        how="left", predicate="within")
    assert matched.plot_id.is_unique, "统计区重叠，请检查边界版本"
    lookup = matched.set_index("plot_id")
    metrics[field] = metrics.plot_id.map(lookup[field])
    metrics[field + "_GEOID"] = metrics.plot_id.map(lookup["GEOID"])
display(metrics.drop(columns="geometry").describe(include="all"))


## 5. 补充建筑／交通等真实指标

前两节仅实现了计数与统计区关联。完成至少5个分析来源前，还需要把已核实的建筑覆盖、交通量等字段关联到同一plot_id。可准备`plot_metrics.csv`，其中plot_id唯一，附加每个指标。计算方法、年份、数据版本、测站距离和缺失规则应写入说明。

建筑覆盖宜按建筑与地块交集去重计算；未覆盖地面并非自动可用。交通量应保留观测日期与方向，不用所有时间段合计冒充日均。不要按邻近测站硬填没有覆盖的道路。


In [ ]:
extra_path = RAW / "plot_metrics.csv"
if extra_path.exists():
    extra = pd.read_csv(extra_path, dtype={"plot_id": str})
    assert extra.plot_id.notna().all() and extra.plot_id.is_unique
    collisions = (set(extra.columns) & set(metrics.columns)) - {"plot_id"}
    assert not collisions, f"避免覆盖现有字段：{collisions}"
    metrics = metrics.merge(extra, on="plot_id", how="left", validate="one_to_one")
metrics.drop(columns="geometry").to_csv(OUT / "all_plot_metrics.csv", index=False)
metrics.to_crs(4326).to_file(OUT / "all_enriched_plots.geojson", driver="GeoJSON")
print("已输出全部候选及属性；这不是最终筛选结果。")


## 6. S05：根据真实分布设置逐层规则

先查看直方图、缺失率和数据覆盖，再填写规则。格式如下（仅语法示例，不是建议阈值）：`{'name':'规则说明','column':'真实字段','op':'>=','value':数值,'reason':'阈值依据'}`。

规则采用逐层AND，记录每层数量与首次排除原因。缺失单独标为待核实；不得当作0。若不足11项，应报告结果并复核方法，不自动调整阈值或抽样凑数。不得按邮编和地块面积组织最终catalog。


In [ ]:
RULES = []  # 请在审查数据分布后填写，至少说明每个阈值依据

if not RULES:
    raise ValueError("尚未配置筛选规则：先审查all_plot_metrics.csv与数据覆盖，再填写RULES")

ops = {">=": lambda a,b:a>=b, "<=": lambda a,b:a<=b,
       ">": lambda a,b:a>b, "<": lambda a,b:a<b, "==": lambda a,b:a==b}
audit = metrics.copy()
audit["selection_status"] = "active"
audit["first_reason"] = ""
funnel = [{"stage": "initial", "remaining": len(audit), "excluded": 0, "review": 0}]
for rule in RULES:
    assert rule.get("reason") and rule.get("name"), "规则必须解释依据"
    field, op = rule["column"], rule["op"]
    assert field in audit and op in ops
    active = audit.selection_status.eq("active")
    missing = active & audit[field].isna()
    passed = ops[op](audit[field], rule["value"]).fillna(False)
    failed = active & ~missing & ~passed
    audit.loc[missing, ["selection_status", "first_reason"]] = ["review", rule["name"] + ": missing"]
    audit.loc[failed, ["selection_status", "first_reason"]] = ["excluded", rule["name"]]
    funnel.append({"stage": rule["name"], "remaining": int(audit.selection_status.eq("active").sum()),
                   "excluded": int(failed.sum()), "review": int(missing.sum())})
selected = audit.loc[audit.selection_status.eq("active")].copy()
pd.DataFrame(funnel).to_csv(OUT / "selection_funnel.csv", index=False)
audit.drop(columns="geometry").to_csv(OUT / "selection_audit.csv", index=False)
(OUT / "selection_rules.json").write_text(json.dumps(RULES, ensure_ascii=False, indent=2))
display(pd.DataFrame(funnel))
print("最终plots:", len(selected))
if len(selected) < 11:
    print("注意：少于任务书要求的>10个Items，需复核数据与筛选方法并如实报告。")


## 7. S06：输出GeoJSON、Rhino与Grasshopper属性表

保留原始地理位置，输出所有多边形部分和内洞边界。每条曲线记录plot_id与GUID，CSV用分号分隔。Multipart的多个曲线不算多个Items。本节输出用于Index的真实位置几何；Items排版应另复制变换，并按你们的分类字段组织。


In [ ]:
if selected.empty:
    raise ValueError("没有通过筛选的plot，不生成空模型")
selected.to_crs(4326).to_file(OUT / "selected_plots.geojson", driver="GeoJSON")
model = rhino3dm.File3dm()
model.Settings.ModelUnitSystem = rhino3dm.UnitSystem.Feet
layer_idx = model.Layers.AddLayer("selected_plots", (20, 120, 180, 255))
records = []
for _, row in selected.iterrows():
    parts = list(row.geometry.geoms) if row.geometry.geom_type == "MultiPolygon" else [row.geometry]
    for part_id, poly in enumerate(parts):
        for ring_id, ring in enumerate([poly.exterior, *poly.interiors]):
            curve = rhino3dm.Polyline([rhino3dm.Point3d(float(p[0]), float(p[1]), 0)
                                       for p in ring.coords]).ToPolylineCurve()
            attrs = rhino3dm.ObjectAttributes()
            attrs.LayerIndex = layer_idx
            attrs.Name = row.plot_id
            attrs.SetUserString("plot_id", row.plot_id)
            guid = model.Objects.Add(curve, attrs)
            record = row.drop(labels=["geometry"]).to_dict()
            record.update(guid=str(guid), part_id=part_id, ring_id=ring_id)
            records.append(record)
assert model.Write(str(OUT / "selected_plots.3dm"), 7), "Rhino写入失败"
pd.DataFrame(records).to_csv(OUT / "selected_plots_attributes.csv", sep=";", index=False)
print("结果目录:", OUT)
print("Rhino与CSV已对应；请人工检查模型单位和边界。")


## 提交前核对

- Index：研究范围、分类schema、最终plots分布、汇总统计和逐层筛选数量。
- Items：>10个真实plots，统一字段与图示，可对应回Index。
- 数据：至少5个具有实际分析用途的来源，列出日期、年份、关联方法与局限。
- 不将空间关联产生的重复行或MultiPolygon部件算成新plot。
- 本框架尚未在你们真实数据上运行；下载、阈值和最终图录仍待完成。
